# Предсказание стоимости жилья

В проекте нам нужно обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году. На основе данных нужно предсказать медианную стоимость дома в жилом массиве. Обучим модель и сделаем предсказания на тестовой выборке. Для оценки качества модели используем метрики RMSE, MAE и R2.

## Описание проекта

В проекте нам нужно обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году. С этим датасетом мы уже работали в четвёртой теме курса. 

В колонках датасета содержатся следующие данные:

longitude — долгота;

latitude — широта;

housing_median_age — медианный возраст жителей жилого массива;

total_rooms — общее количество комнат в домах жилого массива;

total_bedrooms — общее количество спален в домах жилого массива;

population — количество человек, которые проживают в жилом массиве;

households — количество домовладений в жилом массиве;

median_income — медианный доход жителей жилого массива;

median_house_value — медианная стоимость дома в жилом массиве;

ocean_proximity — близость к океану.

На основе данных нужно предсказать медианную стоимость дома в жилом массиве — median_house_value. 
Обучим модель и сделаем предсказания на тестовой выборке. Для оценки качества модели используем метрики RMSE, MAE и R2.


In [ ]:
import pandas as pd
import pyspark
import re
import seaborn as sns
import matplotlib.pyplot as plt

from operator import and_
from functools import reduce

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import Window

import pyspark.sql.functions as F

from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.pipeline import Pipeline

Импорт библиотек прошел удачно.

In [ ]:
RANDOM_SEED = 42

## Инициализируем локальную Spark-сессию.



In [ ]:
spark = SparkSession.builder.master("local").appName("Predicting the cost of housing").getOrCreate()

локальная спарк сессия активирована. Потом выключим

## Прочитаем содержимое файла /datasets/housing.csv.



In [ ]:
data = spark.read.csv('/datasets/housing.csv', inferSchema = True, header=True).drop('_c0')

Прочтено. Ошибок нет.

# Подготовка данных

## Выведем типы данных колонок датасета. Используем методы pySpark.



In [ ]:
data.printSchema()
data.describe().toPandas() 

Уже видно, какие данные какой тип имеют. Категориальный только 1.

## Выполним предобработку данных:



In [ ]:
data = spark.read.csv('/datasets/housing.csv', inferSchema=True, header=True).drop('_c0')
data.show(10)

### Исследуем данные на наличие пропусков и заполним их, выбрав значения по своему усмотрению.



In [ ]:
for y in data.columns:
    print(y, data.where(F.isnan(y) | F.col(y).isNull()).count())

In [ ]:
#207 строчек слишком мало, что бы их отсутствие повлияло на финальный результат. Удалить
#data = data.na.drop(how='any')


In [ ]:
median_total_bedrooms = data.approxQuantile("total_bedrooms", [0.5], 0.001)[0]

data = data.fillna({"total_bedrooms": median_total_bedrooms})

In [ ]:
#категориальные признаки
categorical_features = ['ocean_proximity']
#числовые признаки ('ocean_proximity_numerical')
numerical_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
#целевой признак
target_feature = 'median_house_value'

Разделили признаки для дальнейшей работы с моделями. Но для начала преобразуем парочку.

## Преобразуем колонку с категориальными значениями техникой One hot encoding.



Применим StringIndexer для перевода категориальных признаков в числовые.

In [ ]:
indexer = StringIndexer(inputCols=categorical_features, outputCols=[c+'_numerical' for c in categorical_features]) 
data = indexer.fit(data).transform(data)

А теперь OHE.

In [ ]:
ohe = OneHotEncoder(inputCols=[c+'_numerical' for c in categorical_features], outputCols=[c+'_ohe' for c in categorical_features])

train_data, test_data = data.randomSplit([0.8, 0.2], seed = RANDOM_SEED) #вот

data = ohe.fit(train_data).transform(data)
data.toPandas().head()
data.show(5)

In [ ]:
data.select('ocean_proximity').distinct().collect()

In [ ]:
data.select('ocean_proximity_numerical').distinct().collect()

Красота.


Объединим признаки в один вектор с помощью VectorAssembler:

In [ ]:
categorical_features_assembler = VectorAssembler(inputCols=[c+'_ohe' for c in categorical_features], outputCol="categorical_features")

data = categorical_features_assembler.transform(data)

Преобразование числовых. Ведь для них тоже нужна трансформация. Выбросы мы не убирали.
Соберём вектор числовых признаков в отдельный столбец.

In [ ]:
numerical_features_assembler = VectorAssembler(inputCols=numerical_features, outputCol="numerical_features")
data = numerical_features_assembler.transform(data)
standardScaler = StandardScaler(inputCol='numerical_features',outputCol="plus_numerical_features")
data = standardScaler.fit(data).transform(data) 

In [ ]:
all_features = ['categorical_features','plus_numerical_features'] #это пригодится для задания со всеми признаками и только числовыми
final_assembler = VectorAssembler(inputCols=all_features, outputCol="features") 
data = final_assembler.transform(data)

In [ ]:
sample_data = data.sample(fraction=0.01).limit(1000)
pandas_sample = sample_data.toPandas()

corr_matrix = pandas_sample.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Корреляционная матрица (выборка)')
plt.show()

# Обучение моделей

## Построим две модели линейной регрессии на разных наборах данных:

Для построения модели используем оценщик LinearRegression из библиотеки MLlib.

### используя все данные из файла;

In [ ]:
train, test = data.randomSplit([.8,.2], seed=RANDOM_SEED)


In [ ]:
linear_regression_1 = LinearRegression(labelCol = target_feature, featuresCol='features',regParam=0.0)

In [ ]:
model_1 = linear_regression_1.fit(train) 

In [ ]:
predictions = model_1.transform(test)

In [ ]:
predictions_col = 'prediction'

### используя только числовые переменные, исключив категориальные.

In [ ]:
linear_regression_2 = LinearRegression(labelCol = target_feature, featuresCol='plus_numerical_features', regParam=0.0)


In [ ]:
model_2 = linear_regression_2.fit(train) 

In [ ]:
predictions2 = model_2.transform(test)

In [ ]:
evaluator = RegressionEvaluator(predictionCol = predictions_col, labelCol = target_feature)

# Анализ результатов

## Сравним результаты работы линейной регрессии на двух наборах данных по метрикам RMSE, MAE и R2. 

Сделаем выводы.

In [ ]:
rmse_train = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
rmse_test = evaluator.evaluate(predictions2, {evaluator.metricName: "rmse"})

mae_train = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})
mae_test = evaluator.evaluate(predictions2, {evaluator.metricName: "mae"})

r2_train = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})
r2_test = evaluator.evaluate(predictions2, {evaluator.metricName: "r2"})

results_df = pd.DataFrame({
    'Метрика': ['RMSE', 'MAE', 'R2'],
    'Выборка со всеми признаками': [rmse_train, mae_train, r2_train],
    'Выборка с количественные признаки': [rmse_test, mae_test, r2_test]
})

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=results_df.values,
                 colLabels=results_df.columns,
                 loc='center',
                 cellLoc='center',
                 rowLoc='center')

table.auto_set_font_size(False)
table.set_fontsize(12)
table.auto_set_column_width(col=list(range(len(results_df.columns))))

plt.title("Результаты оценки модели", fontsize=14)
plt.show()

Результаты показали, что модель, обученная на полном наборе данных, превосходит модель, использующую только числовые признаки.
• Лучшая модель — линейная регрессия на полном датасете:

  • RMSE: 70786.46

  • MAE: 50864.55

  • R2: 0.637

Мы проверили результаты на обучающих и тестовых наборах. На тестовом наборе все метрики немного ухудшились, что свидетельствует о том, что модели не переобучились.


In [ ]:
spark.stop()

## Вывод

В ходе работы мы:
Инициализировали локальную Spark-сессию
провели предобработку данных (Исползуя pySpark. Непривычно).
Убрали пропуски, заменив  их медианой.
Обучили две модели линейной регрессии (на полном и на числовых).

В данном проекте была обучена модель 'линейная регрессия' на данных о домах в Калифорнии в 1990 году. Мы освоили инициализацию Spark-сессий, работу с DataFrame API и библиотекой MLlib. В целом модели получились нормальными. И результаты тоже. 

Я бы, конечно, исключил из моделей некоторые признаки, которые, судя по хетмапу, совсем не корелируют с ключевым признаком.